In [29]:
import numpy as np
import pandas as pd
import json
import random
from shapely.geometry import shape, Point
from pathlib import Path

In [30]:
df= pd.read_csv("households.csv")
df

,household_id,lat,lon,district,sector,is_urban,children_under5,avg_meal_count,water_source,sanitation_tier,income_band
0,HH00001,-1.935914,30.047839,Nyarugenge,Kanyinya,1,2,2.5,piped_indoor,unimproved_latrine,Q4
1,HH00002,-2.047530,30.076389,Nyarugenge,Gitega,1,3,2.5,piped_indoor,flush_toilet,Q2
2,HH00003,-1.987029,30.011833,Nyarugenge,Mageragere,1,1,2.2,piped_indoor,improved_latrine,Q3
3,HH00004,-2.026039,30.021708,Nyarugenge,Kimisagara,1,2,2.7,piped_indoor,improved_latrine,Q2
4,HH00005,-1.936134,30.075876,Nyarugenge,Kimisagara,1,3,2.4,piped_yard,improved_latrine,Q2
...,...,...,...,...,...,...,...,...,...,...,...
2495,HH02496,-1.851739,29.943077,Gicumbi,Miyove,0,3,2.3,unprotected_spring,unimproved_latrine,Q2
2496,HH02497,-1.544211,30.069075,Gicumbi,Manyagiro,0,1,1.3,unprotected_spring,open_defecation,Q3
2497,HH02498,-1.665293,29.884544,Gicumbi,Ruvune,0,4,1.0,river_lake,open_defecation,Q5_highest
2498,HH02499,-1.517702,30.292255,Gicumbi,Mutete,0,2,1.8,river_lake,unimproved_latrine,Q4


In [31]:
DISTRICTS = {
    "Nyarugenge": {
        "stunting_baseline": 0.14,  # urban, lower
        "urban_ratio": 0.84,
        "sectors": ["Gitega", "Kanyinya", "Kigali", "Kimisagara", "Mageragere", "Muhima", "Nyakabanda", "Nyamirambo", "Nyarugenge", "Rwezamenyo"],
    },
    "Gasabo": {
        "stunting_baseline": 0.16,
        "urban_ratio": 0.71,
        "sectors": ["Bumbogo", "Gatsata", "Gikomero", "Gisozi", "Jabana", "Jali", "Kacyiru", "Kimihurura", "Kimironko", "Kinyinya", 
                    "Ndera", "Nduba", "Remera", "Rusororo", "Rutunga"],
    },
    "Kicukiro": {
        "stunting_baseline": 0.18,
        "urban_ratio": 0.99,
    "sectors": ["Kicukiro", "Kagarama", "Niboye", "Gatenga", "Gikondo", "Gahanga", "Kanombe", "Nyarugunga", "Kigarama", "Masaka"],
    },
    "Bugesera": {
        "stunting_baseline": 0.28,  # rural, higher
        "urban_ratio": 0.40,
        "sectors": ["Gashora", "Juru", "Kamabuye", "Ntarama", "Mareba", "Mayange", "Musenyi", "Mwogo", "Ngeruka", "Nyamata", "Nyarugenge", "Rilima", 
                    "Ruhuha", "Rweru", "Shyara"],
    },
    "Gicumbi": {
        "stunting_baseline": 0.32,  
        "urban_ratio": 0.06,
        "sectors": ["Bukure", "Bwisige", "Byumba", "Cyumba", "Giti", "Kaniga", "Manyagiro", "Miyove", "Kageyo", "Mukarange", "Muko", "Mutete", "Nyamiyaga", 
                    "Nyankenke II", "Rubaya", "Rukomo", "Rushaki", "Rutare", "Ruvune", "Rwamiko", "Shangasha"],
    },
}

In [32]:
# ─── 2. CATEGORICAL ENCODINGS ─────────────────────────────────────────────────
WATER_SOURCES = ["piped_indoor", "piped_yard", "protected_spring", "unprotected_spring", "river_lake"]
WATER_RISK    = {"piped_indoor": 0.0, "piped_yard": 0.1, "protected_spring": 0.25,
                 "unprotected_spring": 0.55, "river_lake": 0.9}

SANITATION_TIERS = ["flush_toilet", "improved_latrine", "unimproved_latrine", "open_defecation"]
SANITATION_RISK  = {"flush_toilet": 0.0, "improved_latrine": 0.2,
                    "unimproved_latrine": 0.6, "open_defecation": 1.0}

INCOME_BANDS = ["Q1_lowest", "Q2", "Q3", "Q4", "Q5_highest"]
INCOME_RISK  = {"Q1_lowest": 1.0, "Q2": 0.75, "Q3": 0.5, "Q4": 0.25, "Q5_highest": 0.0}

In [33]:
def stunting_probability(row, district_baseline):
    """
    Logistic-style score combining:
      - avg_meal_count  (most important driver)
      - water_source risk
      - sanitation_tier risk
      - income_band risk
      - children_under5 (crowding proxy)
      - district baseline
    """
    meal_risk = max(0.0, (3.0 - row["avg_meal_count"]) / 2.0)   # 0→best, 1→worst
    water_r = WATER_RISK[row["water_source"]]
    sanit_r = SANITATION_RISK[row["sanitation_tier"]]
    income_r = INCOME_RISK[row["income_band"]]
    crowd_r = min(1.0, (row["children_under5"] - 1) / 4.0)

    # Weighted linear combination
    score = (
        0.30 * meal_risk +
        0.25 * water_r +
        0.20 * sanit_r +
        0.15 * income_r +
        0.10 * crowd_r
    )

    # Blend with district baseline
    blended = 0.6 * score + 0.4 * district_baseline

    # Add noise
    noise = np.random.normal(0, 0.05)
    return float(np.clip(blended + noise, 0.01, 0.99))

def assign_stunting_flags(df):
    probs = []
    for _, row in df.iterrows():
        baseline = DISTRICTS[row["district"]]["stunting_baseline"]
        probs.append(stunting_probability(row, baseline))

    df = df.copy()
    df["stunting_risk"] = probs
    
    

    return df

In [34]:
df2= assign_stunting_flags(df)
df2

,household_id,lat,lon,district,sector,is_urban,children_under5,avg_meal_count,water_source,sanitation_tier,income_band,stunting_risk
0,HH00001,-1.935914,30.047839,Nyarugenge,Kanyinya,1,2,2.5,piped_indoor,unimproved_latrine,Q4,0.265865
1,HH00002,-2.047530,30.076389,Nyarugenge,Gitega,1,3,2.5,piped_indoor,flush_toilet,Q2,0.248387
2,HH00003,-1.987029,30.011833,Nyarugenge,Mageragere,1,1,2.2,piped_indoor,improved_latrine,Q3,0.158316
3,HH00004,-2.026039,30.021708,Nyarugenge,Kimisagara,1,2,2.7,piped_indoor,improved_latrine,Q2,0.151801
4,HH00005,-1.936134,30.075876,Nyarugenge,Kimisagara,1,3,2.4,piped_yard,improved_latrine,Q2,0.349656
...,...,...,...,...,...,...,...,...,...,...,...,...
2495,HH02496,-1.851739,29.943077,Gicumbi,Miyove,0,3,2.3,unprotected_spring,unimproved_latrine,Q2,0.440059
2496,HH02497,-1.544211,30.069075,Gicumbi,Manyagiro,0,1,1.3,unprotected_spring,open_defecation,Q3,0.493911
2497,HH02498,-1.665293,29.884544,Gicumbi,Ruvune,0,4,1.0,river_lake,open_defecation,Q5_highest,0.621799
2498,HH02499,-1.517702,30.292255,Gicumbi,Mutete,0,2,1.8,river_lake,unimproved_latrine,Q4,0.340778


In [36]:
df_sector = df2.groupby("sector")["stunting_risk"].agg(["mean", "min", "max", "count"]).reset_index()
df_sector

,sector,mean,min,max,count
0,Bukure,0.452534,0.316014,0.623441,15
1,Bumbogo,0.285918,0.103577,0.609612,46
2,Bwisige,0.437763,0.257477,0.614500,23
3,Byumba,0.433040,0.189560,0.613521,31
4,Cyumba,0.418092,0.185807,0.709656,18
...,...,...,...,...,...
65,Rwamiko,0.407561,0.274244,0.584733,25
66,Rweru,0.368744,0.192528,0.700848,26
67,Rwezamenyo,0.164564,0.052748,0.337702,32
68,Shangasha,0.403711,0.140922,0.666829,28


In [37]:
df_district = df2.groupby("district")["stunting_risk"].agg(["mean"]).reset_index()
df_district

,district,mean
0,Bugesera,0.350081
1,Gasabo,0.254704
2,Gicumbi,0.425167
3,Kicukiro,0.222084
4,Nyarugenge,0.219495


In [ ]:
df_district.to_csv("risk per district.csv", index=False)

In [43]:
df2["rank"] = (
    df2.groupby(["district", "sector"])["stunting_risk"]
      .rank(method="first", ascending=False)
)

df_top10 = df2[df2["rank"] <= 10]
df_top10

,household_id,lat,lon,district,sector,is_urban,children_under5,avg_meal_count,water_source,sanitation_tier,income_band,stunting_risk,rank
4,HH00005,-1.936134,30.075876,Nyarugenge,Kimisagara,1,3,2.4,piped_yard,improved_latrine,Q2,0.349656,4.0
8,HH00009,-1.953736,29.968946,Nyarugenge,Kanyinya,1,5,2.3,piped_indoor,unimproved_latrine,Q4,0.339657,4.0
17,HH00018,-1.758776,29.895891,Nyarugenge,Rwezamenyo,0,2,2.1,protected_spring,improved_latrine,Q1_lowest,0.323743,2.0
22,HH00023,-2.171240,29.940533,Nyarugenge,Nyakabanda,0,1,2.3,river_lake,unimproved_latrine,Q2,0.386657,5.0
23,HH00024,-2.038828,30.067666,Nyarugenge,Kimisagara,1,4,2.1,piped_yard,flush_toilet,Q3,0.325322,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2492,HH02493,-1.500804,30.282052,Gicumbi,Nyamiyaga,0,1,1.3,unprotected_spring,open_defecation,Q2,0.519646,4.0
2493,HH02494,-1.429816,30.127687,Gicumbi,Rushaki,0,2,1.6,river_lake,unimproved_latrine,Q3,0.568880,3.0
2495,HH02496,-1.851739,29.943077,Gicumbi,Miyove,0,3,2.3,unprotected_spring,unimproved_latrine,Q2,0.440059,7.0
2496,HH02497,-1.544211,30.069075,Gicumbi,Manyagiro,0,1,1.3,unprotected_spring,open_defecation,Q3,0.493911,2.0


In [44]:
for district, data in df_top10.groupby("district"):
    filename = f"{district}.csv"
    data.to_csv(filename, index=False)

In [49]:
import pandas as pd
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib import colors

# Load CSV
df = pd.read_csv("Kicukiro.csv")

# Convert dataframe to list (including header)
data = [df.columns.tolist()] + df.values.tolist()

# Create PDF
pdf = SimpleDocTemplate("Kicukiro.pdf")
table = Table(data)

# Style the table
style = TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.grey),
    ('TEXTCOLOR',(0,0),(-1,0),colors.whitesmoke),
    ('GRID', (0,0), (-1,-1), 1, colors.black)
])
table.setStyle(style)

pdf.build([table])